In [5]:
import torch
from model import UNet
from diffusion import DDPM
from dataset import ICLEVRDataset

def run_sanity_check():
    device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.mps.is_available() else "cpu")
    print(f"Testing on: {device}")

    # 1. Model & Diffusion initialization
    model = UNet(base_channels=64).to(device)
    diffusion = DDPM(model, timesteps=1000).to(device)
    
    # 2. Dummy data simulation (Batch size 4, 3 channels, 64x64)
    x_dummy = torch.randn(4, 3, 64, 64).to(device)
    c_dummy = torch.zeros(4, 24).to(device)
    for i in range(4):
        c_dummy[i, torch.randint(0, 24, (3,))] = 1.0 # random 3 objects
        
    print("--- Training Step Pass ---")
    try:
        loss = diffusion.compute_loss(x_dummy, c_dummy, p_uncond=0.1)
        loss.backward()
        print(f"Loss computed: {loss.item():.4f} - Backward pass successful.")
    except Exception as e:
        print(f"Training step FAILED: {e}")
        return

    print("\n--- Inference (DDIM + CFG) Pass ---")
    try:
        # Testing fast sampling with 5 steps only for sanity
        samples = diffusion.ddim_sample(
            shape=(2, 3, 64, 64), 
            context=c_dummy[:2], 
            ddim_steps=5, 
            cfg_scale=3.0
        )
        print(f"Sampling output shape: {samples.shape} - Match expected.")
        
        if torch.isnan(samples).any():
            print("WARNING: NaNs detected in output!")
        else:
            print("Output values are stable.")
    except Exception as e:
        print(f"Inference step FAILED: {e}")
        return

    print("\nSanity Check PASSED. Architecture is ready for training.")

if __name__ == "__main__":
    run_sanity_check()

Testing on: mps
--- Training Step Pass ---
Training step FAILED: Expected weight to be a vector of size equal to the number of channels in input, but got weight of shape [512] and input of shape [4, 384, 16, 16]
